In [1]:
import urllib.request; exec(urllib.request.urlopen('https://aic-data.aiffel.io/api/colab/setup?t=3vvc47bz').read().decode())

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



⏳  터널 준비 확인 중...

✅  터널 생성 완료!
🔗  URL: https://collector-argument-pointer-insured.trycloudflare.com

아래 [URL 복사] 버튼을 누른 뒤 웹앱 연결창에 붙여넣으세요. (이 탭은 열어두세요)


✅ 웹앱에 자동 연결 요청을 보냈습니다. 잠시 후 웹앱 화면이 연결됩니다.


In [2]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn -q

import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False   # 마이너스 부호 깨짐 방지
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.0.2
pandas: 2.2.2


In [3]:
# ─────────────────────────────────────────────
# 모두마켓 데이터 생성 — 이 셀 하나로 오늘 쓸 세 테이블이 모두 준비됩니다.
# 오늘의 초점은 '병합'이므로, 날짜는 분석 가능한 형태(datetime)로 깔끔하게 둡니다.
# (문자열·날짜 오염 정제는 다음 시간 D+005의 주제입니다.)
# ─────────────────────────────────────────────
np.random.seed(42)

# 1) 고객(customers) — 고객 1명이 한 행
n_customers = 300
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_customers + 1)],
    "age": np.random.normal(35, 9, n_customers).round().astype(int).clip(18, 70),
    "gender": np.random.choice(["M", "F"], n_customers),
    "region": np.random.choice(["서울", "경기", "부산", "인천", "대구"], n_customers),
    "membership": np.random.choice(["basic", "premium", "vip"], n_customers, p=[0.6, 0.3, 0.1]),
})

# 2) 상품(products) — 상품 1종이 한 행
categories = ["패션", "뷰티", "식품", "가전", "도서"]
n_products = 40
products = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_products + 1)],
    "category": np.random.choice(categories, n_products),
    "price": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900], n_products),
})

# 3) 주문(orders) — 주문 1건이 한 행. customer_id·product_id로 위 두 표와 연결됩니다.
n_orders = 2000
order_customer = np.random.choice(customers["customer_id"], n_orders)
order_product = np.random.choice(products["product_id"], n_orders)
price_map = products.set_index("product_id")["price"]
quantity = np.random.choice([1, 1, 1, 2, 2, 3], n_orders)
amount = price_map.loc[order_product].values * quantity
order_dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 120, n_orders), unit="D")

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_orders + 1)],
    "customer_id": order_customer,
    "product_id": order_product,
    "quantity": quantity,
    "amount": amount.astype(float),
    "channel": np.random.choice(["web", "app"], n_orders, p=[0.5, 0.5]),
    "order_date": order_dates,
})

print("모두마켓 데이터 생성 완료")
print("customers:", customers.shape, "| products:", products.shape, "| orders:", orders.shape)

모두마켓 데이터 생성 완료
customers: (300, 5) | products: (40, 3) | orders: (2000, 7)


In [4]:
# 세 테이블의 첫인상을 잡아봅니다. 어떤 컬럼이 공통(키)인지 눈으로 찾아보세요.
print("=== orders (주문) — customer_id, product_id 두 개의 키를 가집니다 ===")
display(orders.head())
print("\n=== customers (고객) — customer_id가 키 ===")
display(customers.head(3))
print("\n=== products (상품) — product_id가 키 ===")
display(products.head(3))

=== orders (주문) — customer_id, product_id 두 개의 키를 가집니다 ===


,order_id,customer_id,product_id,quantity,amount,channel,order_date
0,O00001,C0040,P017,1,19900.0,web,2025-01-03
1,O00002,C0224,P022,1,89900.0,app,2025-01-26
2,O00003,C0115,P034,1,49900.0,app,2025-02-26
3,O00004,C0186,P029,1,89900.0,web,2025-01-24
4,O00005,C0056,P004,3,149700.0,web,2025-02-22



=== customers (고객) — customer_id가 키 ===


,customer_id,age,gender,region,membership
0,C0001,39,M,경기,premium
1,C0002,34,F,부산,basic
2,C0003,41,F,서울,premium



=== products (상품) — product_id가 키 ===


,product_id,category,price
0,P001,가전,89900
1,P002,식품,9900
2,P003,도서,9900


In [5]:
# 예제: 월별로 쪼개진 주문 파일을 흉내 내기 — orders를 1월/2월로 나눕니다.
orders_jan = orders[orders["order_date"].dt.month == 1].copy()
orders_feb = orders[orders["order_date"].dt.month == 2].copy()

print("1월 주문:", orders_jan.shape, "| 2월 주문:", orders_feb.shape)

# 세로로 이어 붙이기(axis=0). ignore_index=True로 인덱스를 0부터 다시 매깁니다.
orders_1_2 = pd.concat([orders_jan, orders_feb], axis=0, ignore_index=True)
print("합친 결과:", orders_1_2.shape, "  ← 1월 행 수 + 2월 행 수")
display(orders_1_2.head(3))

1월 주문: (545, 7) | 2월 주문: (435, 7)
합친 결과: (980, 7)   ← 1월 행 수 + 2월 행 수


,order_id,customer_id,product_id,quantity,amount,channel,order_date
0,O00001,C0040,P017,1,19900.0,web,2025-01-03
1,O00002,C0224,P022,1,89900.0,app,2025-01-26
2,O00004,C0186,P029,1,89900.0,web,2025-01-24


In [6]:
# 예제: keys로 '어느 달에서 왔는지' 라벨 달기
# 합친 뒤 출처를 구분해야 할 때 유용합니다.
labeled = pd.concat(
    [orders_jan, orders_feb],
    keys=["2025-01", "2025-02"],   # 바깥 인덱스로 출처 라벨이 붙습니다.
    names=["월", None],
)
print("바깥 인덱스(월)로 출처가 구분됩니다:")
display(labeled.head(3))

# 각 달이 몇 건인지 바깥 인덱스로 세어보기
print("\n월별 주문 건수:")
print(labeled.groupby(level="월").size())

바깥 인덱스(월)로 출처가 구분됩니다:


order_id customer_id product_id  quantity   amount channel  \
월                                                                      
2025-01 0   O00001       C0040       P017         1  19900.0     web   
        1   O00002       C0224       P022         1  89900.0     app   
        3   O00004       C0186       P029         1  89900.0     web   

          order_date  
월                     
2025-01 0 2025-01-03  
        1 2025-01-26  
        3 2025-01-24


월별 주문 건수:
월
2025-01    545
2025-02    435
dtype: int64


In [7]:
# 스스로 해보자! (1)
# 1) 3월 주문, 4월 주문을 각각 만들어 보세요.
orders_3 = orders[orders["order_date"].dt.month == 3].copy()
orders_4 = orders[orders["order_date"].dt.month == 4].copy()

# 2) 두 표를 concat으로 합쳐 보세요 (ignore_index=True 권장).
orders_3_4 = pd.concat([orders_3, orders_4], axis=0, ignore_index=True)

# 3) 행 수를 비교해 출력해 보세요.
print("3월 주문:", orders_3.shape, "| 4월 주문:", orders_4.shape)
print("합친 결과:", orders_3_4.shape, "  ← 3월 행 수 + 4월 행 수")
display(orders_3_4.head())

3월 주문: (521, 7) | 4월 주문: (499, 7)
합친 결과: (1020, 7)   ← 3월 행 수 + 4월 행 수


,order_id,customer_id,product_id,quantity,amount,channel,order_date
0,O00006,C0040,P005,1,129900.0,web,2025-03-11
1,O00008,C0237,P021,2,19800.0,web,2025-03-01
2,O00012,C0147,P019,1,19900.0,web,2025-03-17
3,O00015,C0059,P026,1,49900.0,app,2025-03-07
4,O00019,C0045,P014,1,9900.0,web,2025-03-31


In [8]:
# 예제: orders + customers를 customer_id로 병합 (left 조인)
# how="left": orders(왼쪽)는 전부 남기고, 각 주문에 해당 고객 정보를 붙입니다.
orders_cust = pd.merge(orders, customers, on="customer_id", how="left")

print("병합 전 orders:", orders.shape, "→ 병합 후:", orders_cust.shape)
print("컬럼이 늘었습니다:", list(orders_cust.columns))
display(orders_cust.head(3))

병합 전 orders: (2000, 7) → 병합 후: (2000, 11)
컬럼이 늘었습니다: ['order_id', 'customer_id', 'product_id', 'quantity', 'amount', 'channel', 'order_date', 'age', 'gender', 'region', 'membership']


,order_id,customer_id,product_id,quantity,amount,channel,order_date,age,gender,region,membership
0,O00001,C0040,P017,1,19900.0,web,2025-01-03,37,M,경기,premium
1,O00002,C0224,P022,1,89900.0,app,2025-01-26,20,M,대구,premium
2,O00003,C0115,P034,1,49900.0,app,2025-02-26,33,M,경기,basic


In [9]:
# 예제: 조인 유형 4종 비교 — 일부러 '짝이 안 맞는' 작은 표로 차이를 봅니다.
left = pd.DataFrame({"id": ["A", "B", "C"], "left_val": [1, 2, 3]})
right = pd.DataFrame({"id": ["B", "C", "D"], "right_val": [20, 30, 40]})

print("왼쪽 표 (id: A,B,C)");  display(left)
print("오른쪽 표 (id: B,C,D)"); display(right)

for how in ["inner", "left", "right", "outer"]:
    merged = pd.merge(left, right, on="id", how=how)
    print(f"\n--- how='{how}' → {len(merged)}행 ---")
    display(merged)

왼쪽 표 (id: A,B,C)


,id,left_val
0,A,1
1,B,2
2,C,3


오른쪽 표 (id: B,C,D)


,id,right_val
0,B,20
1,C,30
2,D,40



--- how='inner' → 2행 ---


...

,id,left_val,right_val
0,A,1,NaN
1,B,2,20.0
2,C,3,30.0



--- how='right' → 3행 ---


,id,left_val,right_val
0,B,2.0,20
1,C,3.0,30
2,D,NaN,40



--- how='outer' → 4행 ---


,id,left_val,right_val
0,A,1.0,NaN
1,B,2.0,20.0
2,C,3.0,30.0
3,D,NaN,40.0


In [10]:
# 스스로 해보자! (2)
# orders_cust 에 products를 product_id로 left 병합해 보세요.
full = pd.merge(orders_cust, products, on="product_id", how="left")

# 병합 후 shape와 컬럼, head(3)를 확인해 보세요.
print("orders + customers + products 병합 후:", full.shape)
print(list(full.columns))
display(full.head(3))

orders + customers + products 병합 후: (2000, 13)
['order_id', 'customer_id', 'product_id', 'quantity', 'amount', 'channel', 'order_date', 'age', 'gender', 'region', 'membership', 'category', 'price']


,order_id,customer_id,product_id,quantity,amount,channel,order_date,age,gender,region,membership,category,price
0,O00001,C0040,P017,1,19900.0,web,2025-01-03,37,M,경기,premium,패션,19900
1,O00002,C0224,P022,1,89900.0,app,2025-01-26,20,M,대구,premium,식품,89900
2,O00003,C0115,P034,1,49900.0,app,2025-02-26,33,M,경기,basic,식품,49900


In [11]:
# 스스로 해보자! (3)
# orders + customers를 customer_id로 left 병합하되
# validate="m:1" 과 indicator=True 를 함께 걸어 보세요.
# result = pd.merge(...)
result = pd.merge(
    orders, customers,
    on="customer_id", how="left",
    validate="m:1", indicator=True,
)

# _merge 열의 분포를 출력해, 모두 'both'인지 확인해 보세요.
print("validate 통과! shape:", result.shape)
print(result["_merge"].value_counts())   # 모두 'both'면 미매칭 0건

validate 통과! shape: (2000, 12)
_merge
both          2000
left_only        0
right_only       0
Name: count, dtype: int64


In [20]:
# 분석용 통합 표를 안전하게 다시 만듭니다 (validate로 관계 확인).
full = (
    orders
    .merge(customers, on="customer_id", how="left", validate="m:1")
    .merge(products, on="product_id", how="left", validate="m:1")
)
print("통합 표 full:", full.shape)

# 예제 1) 지역별 평균 주문 금액 — 한 열, 한 함수
region_mean = full.groupby("region")["amount"].mean().round(0)
print("\n[지역별 평균 주문 금액]")
print(region_mean)

통합 표 full: (2000, 13)

[지역별 평균 주문 금액]
region
경기    78616.0
대구    76316.0
부산    80983.0
서울    80417.0
인천    83677.0
Name: amount, dtype: float64


In [23]:
# 스스로 해보자! (4)
# full을 membership으로 그룹지어, named aggregation으로 세 통계를 내보세요.
membership_summary = (
    full.groupby("membership")
    .agg(
        매출합계=("amount", "sum"),
        평균금액=("amount", "mean"),
        주문건수=("order_id", "count"),
    )
    .round(1)
    .sort_values("매출합계", ascending=False)
)
display(membership_summary)

,매출합계,평균금액,주문건수
membership,,,
basic,96340900.0,79752.4,1208
premium,47063000.0,81003.4,581
vip,16395100.0,77701.9,211


# agg 표현 장점
- 한 개 컬럼만 집계: .sum() (간단함)
- 여러 컬럼/여러 통계: agg() (우월함)
- 결과 컬럼명 관리 필요: agg() (필수)

```
## 여러 통계를 한 번에 계산
df.groupby('ticker').agg(
    총매출=("amount", "sum"),
    평균가격=("price", "mean"),
    거래수=("amount", "count"),
    변동성=("price", "std")
)
```

```
## agg 를 안 쓰고 표현 (길어짐, 같은 문법 반복)
result = pd.DataFrame()  # 빈 DataFrame 생성
result['총매출']  = df.groupby('ticker')['amount'].sum()
result['평균가격'] = df.groupby('ticker')['price'].mean()
result['거래수'] = df.groupby('ticker')['amount'].count()
result['변동성'] = df.groupby('ticker')['price'].std()
```

In [25]:
# groupby 에 mean 을 바로 쓰면 집계된 행에 대해 연산. NaN
full["category_avg"] = full.groupby("category")["amount"].mean()
full["diff_from_avg"] = (full["amount"] - full["category_avg"]).round(0)
print("mean 을 바로 쓰면 집계된 행에 대해 연산. NaN 출력")
display(full[["order_id", "category", "amount", "category_avg", "diff_from_avg"]].head(10))

# 예제 1) 각 주문에 '그 카테고리의 평균 금액'을 컬럼으로 붙이기
full["category_avg"] = full.groupby("category")["amount"].transform("mean").round(0)

# '그 주문이 카테고리 평균보다 큰가' (편차) 컬럼도 추가
full["diff_from_avg"] = (full["amount"] - full["category_avg"]).round(0)

print("transform은 행을 줄이지 않습니다. full 행 수:", len(full), "(여전히 2000)")
display(full[["order_id", "category", "amount", "category_avg", "diff_from_avg"]].head(10))



mean 을 바로 쓰면 집계된 행에 대해 연산. NaN 출력


,order_id,category,amount,category_avg,diff_from_avg
0,O00001,패션,19900.0,NaN,NaN
1,O00002,식품,89900.0,NaN,NaN
2,O00003,식품,49900.0,NaN,NaN
3,O00004,뷰티,89900.0,NaN,NaN
4,O00005,뷰티,149700.0,NaN,NaN
5,O00006,패션,129900.0,NaN,NaN
6,O00007,도서,29900.0,NaN,NaN
7,O00008,뷰티,19800.0,NaN,NaN
8,O00009,뷰티,269700.0,NaN,NaN
9,O00010,식품,149700.0,NaN,NaN


transform은 행을 줄이지 않습니다. full 행 수: 2000 (여전히 2000)


,order_id,category,amount,category_avg,diff_from_avg
0,O00001,패션,19900.0,68815.0,-48915.0
1,O00002,식품,89900.0,109537.0,-19637.0
2,O00003,식품,49900.0,109537.0,-59637.0
3,O00004,뷰티,89900.0,80020.0,9880.0
4,O00005,뷰티,149700.0,80020.0,69680.0
5,O00006,패션,129900.0,68815.0,61085.0
6,O00007,도서,29900.0,54282.0,-24382.0
7,O00008,뷰티,19800.0,80020.0,-60220.0
8,O00009,뷰티,269700.0,80020.0,189680.0
9,O00010,식품,149700.0,109537.0,40163.0


In [26]:
# 스스로 해보자! (5)
full["customer_avg"] = full.groupby("customer_id")["amount"].transform("mean").round(0)

# 자기 고객 평균보다 큰 주문(amount > customer_avg)이 몇 건인지 세어 보세요.
bigger = (full["amount"] > full["customer_avg"]).sum()
print(f"자기 평균보다 큰 주문: {bigger}건 / 전체 {len(full)}건")
display(full[["customer_id", "amount", "customer_avg"]].head())

자기 평균보다 큰 주문: 755건 / 전체 2000건


,customer_id,amount,customer_avg
0,C0040,19900.0,61533.0
1,C0224,89900.0,92100.0
2,C0115,49900.0,76433.0
3,C0186,89900.0,124114.0
4,C0056,149700.0,74750.0


In [34]:
# 예제 2) 시간 축 피벗 — 월별 × 카테고리 매출 (실무에서 가장 흔한 형태)
full["month"] = full["order_date"].dt.to_period("M").astype(str)   # 'YYYY-MM'

month_cat = pd.pivot_table(
    full, index="month", columns="category", values="amount",
    aggfunc="sum", fill_value=0,   # 빈 칸은 0으로
).round(0)
print("[월별 × 카테고리 매출]")
display(month_cat.head(10))

# 예제 3) melt — wide를 다시 long으로 (그래프나 저장에 유리한 형태)
month_cat_reset = month_cat.reset_index()
long_form = month_cat_reset.melt(
    id_vars="month",          # 고정할 열
    var_name="category",      # 펼쳐졌던 열 이름이 들어갈 새 컬럼
    value_name="amount",      # 값이 들어갈 새 컬럼
)
print("wide → long 복원 (앞 6행):")
display(long_form.head(6))

# stack/unstack도 같은 일을 합니다: unstack은 wide로 펼치고, stack은 long으로 접습니다.
print("\nstack()으로 다시 접기 (long 형태, 앞부분):")
display(month_cat.stack().head())
print("\nunstack():")
display(month_cat.unstack().head())

[월별 × 카테고리 매출]


category,가전,도서,뷰티,식품,패션
month,,,,,
2025-01,4789500.0,6519100.0,9231500.0,14877000.0,6662400.0
2025-02,4201800.0,6364500.0,7723400.0,9683200.0,5754700.0
2025-03,4919900.0,5053900.0,8431900.0,16556800.0,8212000.0
2025-04,5009700.0,4046800.0,9182000.0,14856400.0,7722500.0


wide → long 복원 (앞 6행):


,month,category,amount
0,2025-01,가전,4789500.0
1,2025-02,가전,4201800.0
2,2025-03,가전,4919900.0
3,2025-04,가전,5009700.0
4,2025-01,도서,6519100.0
5,2025-02,도서,6364500.0



stack()으로 다시 접기 (long 형태, 앞부분):


month    category
2025-01  가전           4789500.0
         도서           6519100.0
         뷰티           9231500.0
         식품          14877000.0
         패션           6662400.0
dtype: float64


unstack():


category  month  
가전        2025-01    4789500.0
          2025-02    4201800.0
          2025-03    4919900.0
          2025-04    5009700.0
도서        2025-01    6519100.0
dtype: float64

In [31]:
# 스스로 해보자! (6)
member_region = pd.pivot_table(
    full, index="membership", columns="region",
    values="amount", aggfunc="mean", fill_value=0,
).round(0)
display(member_region)

region,경기,대구,부산,서울,인천
membership,,,,,
basic,80622.0,74169.0,80731.0,85657.0,76956.0
premium,76234.0,85365.0,84822.0,65652.0,89606.0
vip,75059.0,60374.0,70329.0,78330.0,88488.0


In [32]:
# ─────────────────────────────────────────────
# 종합 실습용 데이터 — 새 스냅샷 (이 셀부터 단독 실행 가능)
# ─────────────────────────────────────────────
np.random.seed(7)

n_cust = 200
cust = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_cust + 1)],
    "region": np.random.choice(["서울", "경기", "부산", "인천"], n_cust),
    "membership": np.random.choice(["basic", "premium", "vip"], n_cust, p=[0.6, 0.3, 0.1]),
})

cats = ["패션", "뷰티", "식품", "가전"]
n_prod = 30
prod = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    "category": np.random.choice(cats, n_prod),
    "price": np.random.choice([12000, 25000, 40000, 75000], n_prod),
})

n_ord = 1500
oc = np.random.choice(cust["customer_id"], n_ord)
op = np.random.choice(prod["product_id"], n_ord)
qty = np.random.choice([1, 1, 2, 3], n_ord)
amt = prod.set_index("product_id")["price"].loc[op].values * qty
odate = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 150, n_ord), unit="D")
ordr = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_ord + 1)],
    "customer_id": oc, "product_id": op,
    "quantity": qty, "amount": amt.astype(float), "order_date": odate,
})
print("스냅샷 준비 완료 — orders:", ordr.shape, "| customers:", cust.shape, "| products:", prod.shape)

스냅샷 준비 완료 — orders: (1500, 6) | customers: (200, 3) | products: (30, 3)


In [36]:
# 시나리오 1 — 검증 → 병합
# (1) 검증: 룩업 표의 키가 유일한가, 주문의 키가 모두 매칭되는가
print("[병합 전 검증]")
print("  customers 키 중복:", cust["customer_id"].duplicated().sum(), "건")
print("  products  키 중복:", prod["product_id"].duplicated().sum(), "건")
print("  매칭 안 되는 customer_id:", (~ordr["customer_id"].isin(cust["customer_id"])).sum(), "건")
print("  매칭 안 되는 product_id :", (~ordr["product_id"].isin(prod["product_id"])).sum(), "건")

# (2) 병합: validate로 관계 가정(m:1)을 강제. indicator로 매칭 확인.
df = (
    ordr
    .merge(cust, on="customer_id", how="left", validate="m:1")
    .merge(prod, on="product_id", how="left", validate="m:1")
)
print("\n[병합 결과] 행 수:", len(df), "(원본 주문 수와 같으면 폭증 없음)")
display(df.head())

[병합 전 검증]
  customers 키 중복: 0 건
  products  키 중복: 0 건
  매칭 안 되는 customer_id: 0 건
  매칭 안 되는 product_id : 0 건

[병합 결과] 행 수: 1500 (원본 주문 수와 같으면 폭증 없음)


,order_id,customer_id,product_id,quantity,amount,order_date,region,membership,category,price
0,O00001,C0032,P005,3,120000.0,2025-04-12,서울,premium,패션,40000
1,O00002,C0049,P020,1,25000.0,2025-03-09,서울,basic,식품,25000
2,O00003,C0041,P016,3,36000.0,2025-01-27,인천,basic,패션,12000
3,O00004,C0123,P002,3,120000.0,2025-03-26,경기,basic,식품,40000
4,O00005,C0150,P014,3,120000.0,2025-03-16,경기,basic,가전,40000


In [44]:
# 1️⃣ 매출 교차표
pivot_sales = df.pivot_table(index="region", columns="membership", values="amount", aggfunc="sum")

# 3️⃣ 평균 구매액 교차표
pivot_avg = df.pivot_table(index="region", columns="membership", values="amount", aggfunc="mean")

# 4️⃣ 합계(행/열)까지 포함
pivot_with_total = df.pivot_table(
    index="region", columns="membership", values="amount",
    aggfunc="sum", margins=True, margins_name="합계"
)

print("📊 매출 교차표:\n", pivot_sales)
print("\n📊 평균 교차표:\n", pivot_avg)
print("\n📊 합계 포함:\n", pivot_with_total)

# 지역별, 카테고리별 순위
# 지역
print("\n📊 지역 순위")
display(df.groupby('region').agg(
    총매출=('amount', 'sum'),
    주문건수=('amount', 'count')
).sort_values('총매출', ascending=False))
# 카테고리
print("\n📊 지역 순위")
display(df.groupby('category').agg(
    총매출=('amount', 'sum'),
    주문건수=('amount', 'count')
).sort_values('총매출', ascending=False))

# 시나리오 2 — 월별 KPI
df["month"] = df["order_date"].dt.to_period("M").astype(str)

monthly_kpi = (
    df.groupby("month")
    .agg(
        총매출=("amount", "sum"),
        주문건수=("order_id", "count"),
        객단가=("amount", "mean"),
        구매고객수=("customer_id", "nunique"),   # nunique: 고유 고객 수
    )
    .round(0)
)
# 전월 대비 매출 증감률(%)도 추가 — 경영 보고서의 단골 지표
monthly_kpi["매출증감률(%)"] = (monthly_kpi["총매출"].pct_change() * 100).round(1)

print("\n\n📊 월별 KPI 요약표 ")
display(monthly_kpi)

# 각 주문이 그 지역 매출에서 차지하는 비중
df['region_sales_ratio'] = df.groupby('region')['amount'].transform(lambda x: x / x.sum())

print("\n\n📊 각 주문이 그 지역 매출에서 차지하는 비중 (앞 20열)")
display(df[['order_id', 'customer_id', 'region', 'amount', 'region_sales_ratio']].head(20))

📊 매출 교차표:
 membership       basic    premium        vip
region                                      
경기          14594000.0  3226000.0  3325000.0
부산          11531000.0  4857000.0  2366000.0
서울          11914000.0  9244000.0  2322000.0
인천          17372000.0  9675000.0   530000.0

📊 평균 교차표:
 membership         basic       premium           vip
region                                              
경기          66944.954128  54677.966102  77325.581395
부산          58831.632653  58518.072289  59150.000000
서울          53426.008969  66503.597122  66342.857143
인천          59088.435374  60093.167702  58888.888889

📊 합계 포함:
 membership       basic     premium        vip          합계
region                                                   
경기          14594000.0   3226000.0  3325000.0  21145000.0
부산          11531000.0   4857000.0  2366000.0  18754000.0
서울          11914000.0   9244000.0  2322000.0  23480000.0
인천          17372000.0   9675000.0   530000.0  27577000.0
합계          55411000.0  270020

,총매출,주문건수
region,,
인천,27577000.0,464
서울,23480000.0,397
경기,21145000.0,320
부산,18754000.0,319



📊 지역 순위


,총매출,주문건수
category,,
패션,38318000.0,512
식품,22404000.0,398
가전,20248000.0,398
뷰티,9986000.0,192




📊 월별 KPI 요약표 


,총매출,주문건수,객단가,구매고객수,매출증감률(%)
month,,,,,
2025-01,19902000.0,330,60309.0,163,NaN
2025-02,17043000.0,289,58972.0,160,-14.4
2025-03,20035000.0,326,61457.0,156,17.6
2025-04,17602000.0,287,61331.0,156,-12.1
2025-05,16374000.0,268,61097.0,146,-7.0




📊 각 주문이 그 지역 매출에서 차지하는 비중 (앞 20열)


,order_id,customer_id,region,amount,region_sales_ratio
0,O00001,C0032,서울,120000.0,0.005111
1,O00002,C0049,서울,25000.0,0.001065
2,O00003,C0041,인천,36000.0,0.001305
3,O00004,C0123,경기,120000.0,0.005675
4,O00005,C0150,경기,120000.0,0.005675
5,O00006,C0188,서울,36000.0,0.001533
6,O00007,C0065,서울,25000.0,0.001065
7,O00008,C0029,부산,75000.0,0.003999
8,O00009,C0008,인천,75000.0,0.002720
9,O00010,C0044,부산,150000.0,0.007998


# 📝 데이터 분석 보고서

## 1. 분석 개요

### ▸ 대상: 모두마켓 주문 1,500건 (2025-01 ~ 2025-05), 고객·상품 정보 병합
## 📊 매출 교차표
| region   |       basic |     premium |        vip |
|----------|------------:|------------:|-----------:|
| 경기     |  14,594,000 |   3,226,000 |   3,325,000 |
| 부산     |  11,531,000 |   4,857,000 |   2,366,000 |
| 서울     |  11,914,000 |   9,244,000 |   2,322,000 |
| 인천     |  17,372,000 |   9,675,000 |     530,000 |

## 📊 평균 교차표
| region   |       basic |      premium |          vip |
|----------|------------:|-------------:|-------------:|
| 경기     |   66,944.95 |   54,677.97  |   77,325.58  |
| 부산     |   58,831.63 |   58,518.07  |   59,150.00  |
| 서울     |   53,426.01 |   66,503.60  |   66,342.86  |
| 인천     |   59,088.44 |   60,093.17  |   58,888.89  |

## 📊 합계 포함
| region   |       basic |     premium |          vip |        합계 |
|----------|------------:|------------:|-------------:|------------:|
| 경기     |  14,594,000 |   3,226,000 |    3,325,000 |  21,145,000 |
| 부산     |  11,531,000 |   4,857,000 |    2,366,000 |  18,754,000 |
| 서울     |  11,914,000 |   9,244,000 |    2,322,000 |  23,480,000 |
| 인천     |  17,372,000 |   9,675,000 |      530,000 |  27,577,000 |
| 합계     |  55,411,000 |  27,002,000 |    8,543,000 |  90,956,000 |

## 📊 지역 순위
| region   |    총매출 |   주문건수 |
|----------|----------:|-----------:|
| 인천     | 27,577,000 |        464 |
| 서울     | 23,480,000 |        397 |
| 경기     | 21,145,000 |        320 |
| 부산     | 18,754,000 |        319 |

## 📊 카테고리 순위
| category |    총매출 |   주문건수 |
|----------|----------:|-----------:|
| 패션     | 38,318,000 |        512 |
| 식품     | 22,404,000 |        398 |
| 가전     | 20,248,000 |        398 |
| 뷰티     |  9,986,000 |        192 |


## 📊 월별 KPI 요약표
| month | 총매출 | 주문건수 | 객단가 | 구매고객수 | 매출증감률(%) |
|-------|--------|---------|--------|-----------|-------------|
| 2025-01 | 19,902,000 | 330 | 60,309 | 163 | - |
| 2025-02 | 17,043,000 | 289 | 58,972 | 160 | -14.4 |
| 2025-03 | 20,035,000 | 326 | 61,457 | 156 | 17.6 |
| 2025-04 | 17,602,000 | 287 | 61,331 | 156 | -12.1 |
| 2025-05 | 16,374,000 | 268 | 61,097 | 146 | -7.0 |

## 2. 핵심 발견 (위 셀에서 추출한 숫자로 채우세요)

### ▸ 매출이 가장 높았던 달은 3월(다음은 1월) 이였으며, 전월 대비 증감률은 17.6%.
### ▸ 카테고리 중 패션(식품,가전,뷰티 순), 지역 중 인천(서울,부산 순)의 매출 기여가 가장 컸음.
### ▸ 전체 객단가는 약 60,637원.

## 3. 데이터 신뢰성 (반드시 기록)

### ▸ 병합 전 키 검증: 중복 0건, 미매칭 0건 → 행 폭증·조용한 결측 없음.
### ▸ validate="m:1" 통과 → 관계 가정(주문:상품 = m:1) 유효.

## 4. 제언

- 월별 매출 변동의 원인: 3월의 증가는 구매고객수는 같은데, 구매건수가 늘었음.
- 객단가는 2월에 비해 소폭 증가. 하지만, 1,4,5 월도 객단가가 6만원대.
- 1월도 매출이 높은데, 이 때도 구매건수의 증가.
- 현재 데이터로 보면 고객이 지속 구매로 구매건수가 늘었기 때문에 매출이 늘어난 것으로 보임.
- 객단가 증가는 매출에 영향이 없음. 객단가 평균 이상이지만 오히려 하락한 적도 있음
- 따라서, 고객이 지속 구매하는 요인이 있을 것으로 보고 그 원인을 분석한 후 그 결과에 따라 지속 구매 마케팅 전략을 수립하여 공략하는 것이 현재 매출을 빠른 시간에 올릴 효율적 방법일 수 있음.

# 진영 강사님 빡십니다. 헥헥. 문법이 다 안 외워짐.